# Self-supervised YouTube-ASL Skeleton BERT

This notebook clones the code from GitHub and keeps keypoint data and active training files in Colab's temporary `/content` storage. It mounts Google Drive for persistence only: after every completed shard/epoch it copies the latest checkpoints, metrics, and completion state to Drive, verifies the copy, and only then deletes the local data shard. If Drive mounting fails, it falls back to downloading a compact resume bundle. Choose **Runtime → Change runtime type → GPU** first.

This run defaults to `MODE = 'full'`. Full mode downloads and trains one shard at a time, persists resumable checkpoints, and removes each local data shard after it finishes.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 1. Configure Drive-backed checkpoint persistence and clone GitHub

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

if shutil.which('aria2c') is None:
    print('Installing aria2 for multi-connection downloads...')
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)

USE_GOOGLE_DRIVE = True
DRIVE_MOUNTED = False
if USE_GOOGLE_DRIVE:
    COLAB_ACCOUNT_EMAIL = input('Email currently used for Colab: ').strip().casefold()
    TARGET_DRIVE_EMAIL = input('Different email whose Drive should receive checkpoints: ').strip().casefold()
    if not COLAB_ACCOUNT_EMAIL or not TARGET_DRIVE_EMAIL:
        raise ValueError('Both account emails are required')
    if COLAB_ACCOUNT_EMAIL == TARGET_DRIVE_EMAIL:
        raise ValueError('Colab and target Drive accounts must be different')
    print(f'In the Google authorisation dialog, select exactly: {TARGET_DRIVE_EMAIL}')
    try:
        drive.mount('/content/drive')
        marker_root = Path('/content/drive/MyDrive/sign_semantics_youtube_asl')
        marker_root.mkdir(parents=True, exist_ok=True)
        account_marker = marker_root / '.target_drive_account.txt'
        if account_marker.exists():
            recorded_email = account_marker.read_text().strip().casefold()
            if recorded_email != TARGET_DRIVE_EMAIL:
                raise RuntimeError(
                    f'Drive marker belongs to {recorded_email}, not {TARGET_DRIVE_EMAIL}'
                )
        else:
            account_marker.write_text(TARGET_DRIVE_EMAIL + '\n')
        print('Drive account marker verified for:', TARGET_DRIVE_EMAIL)
        DRIVE_MOUNTED = True
    except Exception as error:  # Colab exposes several mount error types
        print('Drive unavailable; browser resume downloads will be used:', error)

REPO_URL = 'https://github.com/ss-sebastian/youtube-asl-skeleton-bert.git'
PROJECT = Path('/content/youtube-asl-skeleton-bert')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
print('Project ready:', PROJECT)

## 2. Select pilot or resumable full training

In [ ]:
MODE = 'full'  # full training over all official shards
PILOT_TRAIN_CLIPS = 512
PILOT_VAL_CLIPS = 128
FULL_SHARDS = list(range(1, 11))
BATCH_SIZE = 16
MAX_FRAMES = 256
NUM_WORKERS = 2
PREFETCH_FACTOR = 2

LOCAL_ROOT = Path('/content/youtube_asl_data')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_PERSIST_ROOT = Path('/content/sign_semantics_youtube_asl')
RUN_ROOT = LOCAL_PERSIST_ROOT / MODE
CHECKPOINT_ROOT = RUN_ROOT / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT = (
    Path('/content/drive/MyDrive/sign_semantics_youtube_asl') / MODE
    if DRIVE_MOUNTED
    else None
)
RESTORED_FROM_DRIVE = False
if DRIVE_RUN_ROOT is not None and (DRIVE_RUN_ROOT / 'completed_shards.json').exists():
    shutil.copytree(DRIVE_RUN_ROOT, RUN_ROOT, dirs_exist_ok=True)
    RESTORED_FROM_DRIVE = True
    print('Restored training state from Drive:', DRIVE_RUN_ROOT)
RESUME_BUNDLE = Path('/content/youtube_asl_full_resume.zip')
if RESUME_BUNDLE.exists() and not RESTORED_FROM_DRIVE:
    with zipfile.ZipFile(RESUME_BUNDLE) as handle:
        handle.extractall(RUN_ROOT)
    print('Restored prior full-run state from:', RESUME_BUNDLE)
print('Data (temporary):', LOCAL_ROOT)
print('Models:', CHECKPOINT_ROOT)
print('Persistence:', 'Google Drive after every shard' if DRIVE_MOUNTED else 'browser resume bundle after every shard')
disk = shutil.disk_usage('/content')
print(f'Colab disk: {disk.free / 1024**3:.1f} GiB free / {disk.total / 1024**3:.1f} GiB total')

## 3. Official dataset URLs and train/dev annotations

In [ ]:
SHARDS = {
 1: ('3dc57bf4-c5fb-491c-8ab2-a9215e0e2fe5', '75223dd5e7e9b6ccb9f34c5792fc1e6d'),
 2: ('bfa38e27-bd46-48ae-bf9a-eb1eafaabb95', 'c891a51901ca17fa6f42529e5657df67'),
 3: ('0c59bda5-908d-4194-93bf-13e13de2ef10', '0713086c142d52c31cff1b4be9f4f82a'),
 4: ('1a4ace36-ed9a-4bfb-a80f-483c0463e02d', '890dd2fa779b56cd90c6ae39fa67faef'),
 5: ('6b405702-9f74-4729-8202-55eca36adaea', 'e935642e8bb5f9b594af74a8ba75d97f'),
 6: ('3b4ef094-bc95-4efb-b8e0-3bc4f63e57b0', '403958646966402245f69f9d473c4346'),
 7: ('74e99da7-c580-4fdf-8363-8024d7a7adf1', '786fe0067d1e4d8665b1ddbb8628a17c'),
 8: ('43a9146e-0abf-47ec-a3b3-90c01a4d9380', 'b6bcc2e8517c2dcf8347347bbd74800c'),
 9: ('05385788-d459-4f35-92b4-4908b9d86de6', '9d52caab1aa2db4218188819485f92ab'),
10: ('d96b874e-72fa-4830-b2f0-a072bb6be31d', '0019b603f9ebd7594fce8fae8dc65167'),
}
BASE = 'https://lindat.mff.cuni.cz/repository/server/api/core/bitstreams'
TRAIN_ANNOTATION_URL = f'{BASE}/f8460818-3605-4f05-9832-90ddc68f22e6/content'
DEV_ANNOTATION_URL = f'{BASE}/d5d23d31-c93a-4752-8e5c-e20548f13da0/content'
TRAIN_ANNOTATIONS = LOCAL_ROOT / 'YT.translations.train.json'
DEV_ANNOTATIONS = LOCAL_ROOT / 'YT.translations.dev.json'

def fast_download(source, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    aria2 = shutil.which('aria2c')
    if aria2:
        command = [
            aria2, '--continue=true', '--max-connection-per-server=16',
            '--split=16', '--min-split-size=16M', '--file-allocation=none',
            '--auto-file-renaming=false', '--allow-overwrite=false',
            '--max-tries=0', '--retry-wait=5', '--summary-interval=10',
            f'--dir={target.parent}', f'--out={target.name}', source,
        ]
    else:
        command = ['wget', '-c', '--show-progress', '-O', str(target), source]
    print('Downloading with:', Path(command[0]).name, target.name, flush=True)
    subprocess.run(command, check=True)

def print_disk(label):
    usage = shutil.disk_usage('/content')
    print(
        f'{label}: used={usage.used / 1024**3:.1f} GiB, '
        f'free={usage.free / 1024**3:.1f} GiB',
        flush=True,
    )

for url, target in [(TRAIN_ANNOTATION_URL, TRAIN_ANNOTATIONS), (DEV_ANNOTATION_URL, DEV_ANNOTATIONS)]:
    if not target.exists():
        fast_download(url, target)
print('Annotations:', TRAIN_ANNOTATIONS.stat().st_size, DEV_ANNOTATIONS.stat().st_size)

## 4. Train

Pilot mode reads limited entries through HTTP range requests. Full mode needs roughly 45 GB free for one compressed shard; it never extracts the ZIP. `completed_shards.json`, checkpoints, and metric files are synced to Google Drive after every shard. The local shard is deleted only after Drive files pass a size check. If Drive is unavailable, keep the automatically downloaded resume bundle; upload it later as `/content/youtube_asl_full_resume.zip` to restore.

In [ ]:
import hashlib
import json


def shard_url(number):
    return f'{BASE}/{SHARDS[number][0]}/content'

def md5(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.md5()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_config(archive, epochs, pilot=False):
    config = json.loads((PROJECT / 'configs/pretrain.json').read_text())
    config['data'].update({
        'train_archive': str(archive),
        'val_archive': str(archive),
        'train_annotations': str(TRAIN_ANNOTATIONS),
        'val_annotations': str(DEV_ANNOTATIONS),
        'max_frames': MAX_FRAMES,
        'num_workers': 0 if pilot else NUM_WORKERS,
        'prefetch_factor': PREFETCH_FACTOR,
    })
    if pilot:
        config['data']['limit_train_clips'] = PILOT_TRAIN_CLIPS
        config['data']['limit_val_clips'] = PILOT_VAL_CLIPS
    config['training'].update({
        'output_dir': str(CHECKPOINT_ROOT),
        'batch_size': BATCH_SIZE,
        'progress_every': 1,
        'epochs': epochs,
        'amp': True,
        'save_every': 0,  # last.pt + best.pt + downloaded resume bundle save disk
    })
    path = Path('/content/colab_pretrain.json')
    path.write_text(json.dumps(config, indent=2))
    return path

def train_once(config_path, resume=None):
    command = [sys.executable, '-u', '-m', 'sign_semantics.train', '--config', str(config_path)]
    if resume is not None and resume.exists():
        command += ['--resume', str(resume)]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

def sync_update_to_drive(state_path=None):
    if not DRIVE_MOUNTED or DRIVE_RUN_ROOT is None:
        return False
    drive_checkpoints = DRIVE_RUN_ROOT / 'checkpoints'
    drive_checkpoints.mkdir(parents=True, exist_ok=True)
    sources = []
    for name in ('last.pt', 'best.pt', 'metrics.jsonl', 'metrics.csv'):
        source = CHECKPOINT_ROOT / name
        if source.exists():
            sources.append((source, drive_checkpoints / name))
    if state_path is not None and state_path.exists():
        sources.append((state_path, DRIVE_RUN_ROOT / state_path.name))
    if not any(source.name == 'last.pt' for source, _ in sources):
        raise RuntimeError('Refusing to sync: last.pt is missing')
    for source, destination in sources:
        temporary = destination.with_name(destination.name + '.uploading')
        shutil.copy2(source, temporary)
        if temporary.stat().st_size != source.stat().st_size:
            raise RuntimeError(f'Drive size check failed for {source.name}')
        temporary.replace(destination)
    print('Drive update verified:', DRIVE_RUN_ROOT, flush=True)
    return True

def download_resume_bundle(state_path, shard):
    bundle = Path(f'/content/youtube_asl_full_resume_after_shard_{shard:02d}.zip')
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as handle:
        handle.write(state_path, 'completed_shards.json')
        for name in ('last.pt', 'best.pt'):
            checkpoint = CHECKPOINT_ROOT / name
            if checkpoint.exists():
                handle.write(checkpoint, f'checkpoints/{name}')
        metrics = CHECKPOINT_ROOT / 'metrics.jsonl'
        if metrics.exists():
            handle.write(metrics, 'checkpoints/metrics.jsonl')
        metrics_csv = CHECKPOINT_ROOT / 'metrics.csv'
        if metrics_csv.exists():
            handle.write(metrics_csv, 'checkpoints/metrics.csv')
    print('Downloading resumable checkpoint bundle:', bundle)
    files.download(str(bundle))
    for older in Path('/content').glob('youtube_asl_full_resume_after_shard_*.zip'):
        if older != bundle:
            older.unlink()
    print_disk('After saving the resume bundle')

def persist_update(state_path, shard):
    if not sync_update_to_drive(state_path):
        download_resume_bundle(state_path, shard)

if MODE == 'pilot':
    config_path = write_config(shard_url(1), epochs=1, pilot=True)
    train_once(config_path)
    if DRIVE_MOUNTED:
        sync_update_to_drive()
else:
    state_path = RUN_ROOT / 'completed_shards.json'
    completed = json.loads(state_path.read_text()) if state_path.exists() else []
    full_run_started = __import__('time').perf_counter()
    session_completed = 0
    for shard in FULL_SHARDS:
        if shard in completed:
            print(f'Shard {shard} already completed; skipping.')
            continue
        print(f'Full progress: {len(completed)}/{len(FULL_SHARDS)} shards; starting shard {shard}.', flush=True)
        archive = LOCAL_ROOT / f'raw_keypoints_{shard}.zip'
        reusable_gb = archive.stat().st_size / 1024**3 if archive.exists() else 0
        available_gb = shutil.disk_usage('/content').free / 1024**3 + reusable_gb
        if available_gb < 45:
            raise RuntimeError(
                f'Need 45 GB free or reusable; only {available_gb:.1f} GB available.'
            )
        print_disk(f'Before shard {shard} download')
        fast_download(shard_url(shard), archive)
        print_disk(f'After shard {shard} download')
        actual_md5 = md5(archive)
        if actual_md5 != SHARDS[shard][1]:
            raise RuntimeError(f'MD5 mismatch for shard {shard}: {actual_md5}')
        config_path = write_config(archive, epochs=len(completed) + 1)
        resume = CHECKPOINT_ROOT / 'last.pt'
        train_once(config_path, resume if completed else None)
        completed.append(shard)
        session_completed += 1
        state_path.write_text(json.dumps(completed))
        persist_update(state_path, shard)
        archive.unlink()
        print_disk(f'After deleting shard {shard} data')
        elapsed = __import__('time').perf_counter() - full_run_started
        stages_this_session = max(1, session_completed)
        remaining = len(FULL_SHARDS) - len(completed)
        eta_hours = elapsed / stages_this_session * remaining / 3600
        print(
            f'Completed shard {shard}; local data removed. Full progress: '
            f'{len(completed)}/{len(FULL_SHARDS)}; session ETA: {eta_hours:.1f} hours.',
            flush=True,
        )

## 5. Verify the local model

In [ ]:
for path in sorted(CHECKPOINT_ROOT.glob('*.pt')):
    print(path, f'{path.stat().st_size / 1024**2:.1f} MB')
assert (CHECKPOINT_ROOT / 'last.pt').exists(), 'Training did not write last.pt'
print('Local model:', CHECKPOINT_ROOT / 'last.pt')
metrics_path = CHECKPOINT_ROOT / 'metrics.jsonl'
if metrics_path.exists():
    import pandas as pd
    metrics = pd.read_json(metrics_path, lines=True)
    display(metrics)
    metrics.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', grid=True)

## 6. Verify the latest Google Drive update

The training loop already updates Drive after every completed shard. This optional cell lists and verifies the latest persistent files. Raw keypoint data is never copied to Drive.

In [ ]:
assert DRIVE_MOUNTED and DRIVE_RUN_ROOT is not None, 'Google Drive is not mounted'
required = [
    DRIVE_RUN_ROOT / 'completed_shards.json',
    DRIVE_RUN_ROOT / 'checkpoints' / 'last.pt',
    DRIVE_RUN_ROOT / 'checkpoints' / 'metrics.jsonl',
    DRIVE_RUN_ROOT / 'checkpoints' / 'metrics.csv',
]
for path in required:
    assert path.exists() and path.stat().st_size > 0, f'Missing or empty: {path}'
    print(path, f'{path.stat().st_size / 1024**2:.2f} MiB')
print('Latest Drive update is complete.')